<h3>Gunjan Gupta</h3>
<h4>Roll No: BEBD23274</h4>

# Practical 7 (B2) - Interacting with Web APIs
### Analyzing Weather Data from OpenWeatherMap API

**Task 1:** Register and obtain API key from OpenWeatherMap.

**Setup:** Import required libraries.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
print("Libraries ready")

**Task 2:** Interact with the OpenWeatherMap API using the API key to retrieve weather data for a specific location.

In [ ]:
API_KEY = "73c47ecfe8d2c015bffd405d35d4b57a"
CITY = "Pune"

url = f"https://api.openweathermap.org/data/2.5/forecast?q={CITY}&appid={API_KEY}&units=metric"

response = requests.get(url)
data = response.json()

print("Status code:", response.status_code)
print("City:", data["city"]["name"])
print("Number of readings received:", len(data["list"]))

**Task 3:** Extract relevant weather attributes such as temperature, humidity, wind speed, and precipitation from the API response.

In [ ]:
records = []

for item in data["list"]:
    records.append({
        "datetime": item["dt_txt"],
        "temperature": item["main"]["temp"],
        "humidity": item["main"]["humidity"],
        "pressure": item["main"]["pressure"],
        "wind_speed": item["wind"]["speed"],
        "precipitation": item.get("rain", {}).get("3h", 0),
        "weather": item["weather"][0]["main"]
    })

df = pd.DataFrame(records)
df.head()

**Task 4:** Clean and preprocess the retrieved data, handling missing values or inconsistent formats.

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])

print("Missing values per column:")
print(df.isnull().sum())

df = df.fillna(0)
df = df.drop_duplicates()

print("\nData types after cleaning:")
print(df.dtypes)
df.head()

**Task 5:** Perform data modeling to analyze weather patterns, such as calculating average temperature, maximum/minimum values, or trends over time.

In [ ]:
print("Average Temperature: %.2f C" % df["temperature"].mean())
print("Maximum Temperature: %.2f C" % df["temperature"].max())
print("Minimum Temperature: %.2f C" % df["temperature"].min())
print("Average Humidity: %.2f %%" % df["humidity"].mean())
print("Average Wind Speed: %.2f m/s" % df["wind_speed"].mean())

df.describe()

**Task 6:** Visualize the weather data using appropriate plots, such as line charts, bar plots, or scatter plots, to represent temperature changes, precipitation levels, or wind speed variations.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["datetime"], df["temperature"], marker="o", color="tomato")
plt.title("Temperature Trend Over Time")
plt.xlabel("Date/Time"); plt.ylabel("Temperature (C)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

plt.figure(figsize=(12, 4))
plt.bar(df["datetime"].astype(str), df["wind_speed"], color="skyblue")
plt.title("Wind Speed Over Time")
plt.xlabel("Date/Time"); plt.ylabel("Wind Speed (m/s)")
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 5))
plt.scatter(df["temperature"], df["humidity"], color="green")
plt.title("Temperature vs Humidity")
plt.xlabel("Temperature (C)"); plt.ylabel("Humidity (%)")
plt.tight_layout(); plt.show()

**Task 7:** Apply data aggregation techniques to summarize weather statistics by specific time periods (e.g., daily, monthly, seasonal).

In [ ]:
df["date"] = df["datetime"].dt.date

daily = df.groupby("date").agg(
    avg_temp=("temperature", "mean"),
    max_temp=("temperature", "max"),
    min_temp=("temperature", "min"),
    avg_humidity=("humidity", "mean"),
    total_rain=("precipitation", "sum")
).reset_index()

daily

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(daily["date"], daily["avg_temp"], marker="s", color="purple")
plt.title("Daily Average Temperature")
plt.xlabel("Date"); plt.ylabel("Avg Temp (C)")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

**Task 8:** Incorporate geographical information, if available, to create maps or geospatial visualizations representing weather patterns across different locations.

In [ ]:
lat = data["city"]["coord"]["lat"]
lon = data["city"]["coord"]["lon"]
print(f"{CITY} is at latitude {lat}, longitude {lon}")

plt.figure(figsize=(6, 6))
plt.scatter(lon, lat, color="red", s=200, marker="*")
plt.text(lon, lat + 0.02, CITY, fontsize=12, ha="center")
plt.title(f"Geographical Location of {CITY}")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.tight_layout()
plt.show()

**Task 9:** Explore and visualize relationships between weather attributes, such as temperature and humidity, using correlation plots or heatmaps.

In [ ]:
numeric_cols = ["temperature", "humidity", "pressure", "wind_speed", "precipitation"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Weather Attributes")
plt.tight_layout(); plt.show()